# Notebook 04 — Tokens and Causal Language Modeling

    ## Learning objectives

    - Distinguish bytes, characters, words, and subword tokens
- Construct shifted labels for next-token prediction
- Relate context length, vocabulary size, logits, and generation

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 4.1 Tokenization is a learned compression interface

A tokenizer maps text to integer IDs from a finite vocabulary. Byte-level BPE and
unigram tokenizers preserve arbitrary text while learning reusable multi-byte pieces.
Token boundaries are not linguistic truth: spelling, whitespace, code, and language
all change token efficiency.

If the vocabulary has size \(V\), the model emits \(V\) logits at every position.
Larger vocabularies shorten sequences but enlarge embedding/output matrices. Smaller
vocabularies do the reverse. Tokenizer choice is therefore part of model architecture.


In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
samples = ["unbelievable", " leading space", "def fib(n):", "日本語の文章", "🧠"]
for text in samples:
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text!r}\n  ids={ids}\n  pieces={pieces}\n")


## 4.2 The causal objective

For tokens \(x_1,\ldots,x_T\), an autoregressive model factorizes

\[
p(x_{1:T}) = \prod_{t=1}^{T} p(x_t \mid x_{<t}).
\]

A single sequence supplies many supervised examples. Inputs are tokens
\([x_1,\ldots,x_{T-1}]\); labels are shifted left
\([x_2,\ldots,x_T]\). A causal mask prevents position \(t\) from attending to future
positions. Padding labels are commonly replaced with `-100` so cross-entropy ignores
them.


In [ ]:
import torch

batch = tokenizer(["Models predict tokens.", "Causal masks prevent leakage."],
                  padding=True, return_tensors="pt")
input_ids = batch["input_ids"]
labels = input_ids.clone()
labels[batch["attention_mask"] == 0] = -100
print("input_ids:\n", input_ids)
print("labels used by a causal LM:\n", labels)
print("The model internally aligns logits[:, :-1] with labels[:, 1:].")


## 4.3 From logits to text

Logits are unnormalized scores. Greedy decoding chooses the largest logit; sampling
draws from a probability distribution after temperature/top-k/top-p transformations.
Training uses teacher forcing, whereas generation consumes the model's own prior
outputs—one reason exposure errors can compound.


## 4.4 How modern subword tokenizers are built

A useful tokenizer must be lossless, reasonably compact, fast, and stable across the
model's target languages and domains. Byte-level BPE begins with bytes, so every input is
representable. It repeatedly merges frequent adjacent symbols until it reaches a target
vocabulary size. WordPiece uses a related greedy vocabulary construction. Unigram begins
with a large candidate vocabulary and removes pieces while minimizing a probabilistic
objective. SentencePiece treats whitespace as an ordinary symbol and can train directly
on raw text.

Vocabulary construction is a distributional decision. A tokenizer trained mainly on
English prose may split another language, source code, chemical notation, or identifiers
inefficiently. That increases sequence length, attention cost, and the number of prediction
steps. Special tokens also carry architectural meaning: BOS/EOS delimit sequences; PAD
aligns batches; chat-control tokens mark roles; tool or multimodal tokens may reserve spans.
Adding tokens after pretraining creates randomly initialized embedding/output rows unless
they are deliberately initialized and trained.

**Reference rule:** never infer token counts from character counts in production. Use the
exact tokenizer revision paired with the exact model revision and chat template.


In [ ]:
# Inspect vocabulary behavior and round-trip invariants.
probes = [
    "hello", " hello", "hello\n", "HTTPResponseFactory", "2.718281828",
    "naïve café", "مرحبا بالعالم", "中文分词", "👩🏽‍💻",
]
for text in probes:
    ids = tokenizer.encode(text, add_special_tokens=False)
    decoded = tokenizer.decode(ids)
    print({"text": text, "tokens": len(ids), "ids": ids,
           "round_trip": decoded == text, "decoded": decoded})

special = tokenizer.special_tokens_map
print("\nspecial tokens:", special)
print("vocabulary size:", tokenizer.vocab_size)


## 4.5 Sequence construction, boundaries, and label masking

Language-model examples are usually concatenated and sliced into fixed-length blocks.
Boundaries matter. Without EOS separators, the model is trained to continue the end of one
document with the beginning of an unrelated one. Packing improves token utilization but
can allow cross-example attention unless a block-diagonal mask is used. During instruction
tuning, training on every role teaches the model to reproduce user text as well as assistant
text; completion-only or assistant-only masks instead put `-100` on non-target positions.

Padding and loss masking solve different problems. The attention mask says which positions
may participate in attention. A `-100` label tells PyTorch cross-entropy not to score that
position. Decoder-only generation is commonly left-padded in batches because generation
begins after the final array position, while training is often right-padded. Inspect rather
than assume a model's pad token: many causal models reuse EOS, which is acceptable only when
masks correctly distinguish padding from actual EOS occurrences.


In [ ]:
# A fully explicit next-token example.
text = "Attention reuses cached keys."
ids = tokenizer.encode(text, add_special_tokens=False)
print("position | input piece -> target piece")
for position, (current, target) in enumerate(zip(ids[:-1], ids[1:])):
    print(f"{position:8d} | {tokenizer.decode([current])!r:15} -> {tokenizer.decode([target])!r}")

# Each row of a causal mask can see itself and earlier positions only.
import torch
T = min(len(ids), 8)
allowed = torch.tril(torch.ones(T, T, dtype=torch.int))
print("\ncausal visibility mask (query rows, key columns):\n", allowed)


## 4.6 Context windows and practical token budgets

A context limit covers prompt, chat-control tokens, retrieved evidence, tool schemas and
results, prior messages, and generated output. Reserving no output budget is a common bug.
The maximum advertised window is also not a promise of equal accuracy at every position.
Long contexts increase prefill work and KV-cache memory, may dilute relevant evidence, and
can suffer position-dependent retrieval failures.

Build token budgeting as a deterministic preprocessing stage: render the final chat
template, count exact tokens, reserve output and safety margin, then apply an explicit
policy—truncate low-priority history, summarize, retrieve fewer chunks, or reject. Silent
truncation can remove the user's question, an assistant target, citations, or image tokens.

**Diagnostic checklist:** record tokenizer ID/revision, rendered prompt token count, special
tokens added, truncation side, content removed, reserved output, and model context limit.
Tokenization bugs often masquerade as model-quality problems.


## 4.7 Tokenization reference and common pitfalls

| Term | Practical meaning |
|---|---|
| Vocabulary | ID-to-piece inventory paired with embedding/output rows |
| Encode/decode | Text→IDs and IDs→text; round-trip behavior may normalize details |
| Special token | Control token with template/model semantics, not ordinary prose |
| Chat template | Deterministic rendering of structured messages into model tokens |
| Attention mask | Controls visible/valid input positions |
| Label mask | Controls which target positions contribute loss, commonly with `-100` |
| Packing | Combining examples to reduce padding waste |

Common failures are counting tokens before applying the chat template, adding a pad/special token
without resizing/training embeddings, using a tokenizer from a similarly named but different model,
double-adding BOS/EOS, decoding the prompt together with generated output, and silently truncating
from the wrong side. Unicode adds subtleties: visually identical strings can use different code-point
normalization; emoji can contain joiners and modifiers; byte fallback preserves inputs but may be
token-inefficient. Log exact token IDs when a format behaves unexpectedly.

**Connection forward:** token length drives training batches, attention cost, KV memory, RAG chunk
sizes, and serving capacity. The tokenizer is not preprocessing plumbing—it is the discrete interface
on which every later notebook depends.


## Exercises

    1. Compare token counts for English, code, and two non-English languages.
2. Show exactly which target each position predicts for a five-token sequence.
3. Explain why changing a tokenizer can invalidate pretrained embedding weights.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
